In [8]:
!pip install -q --upgrade transformers

In [11]:
# Scenario: Fraud Detection in Customer Support Chats
# - Problem: Fraudsters often contact bank support pretending to be customers, trying to reset passwords or gain access.
# - Challenge: Analysts can’t manually read millions of chat transcripts.
# - Solution: Use LoRA fine‑tuning on a pretrained language model (like distilbert-base-uncased) to classify chats as:
# - 0 → Normal inquiry
# - 1 → Fraud attempt
# - 2 → Suspicious but unclear

# Install libraries
!pip install -q transformers datasets peft accelerate scikit-learn

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split

# Step 1: Sample Dataset
data = [
    {"text":"Hello I forgot my password can you help me reset it", "label":0},
    {"text":"What are the bank working hours", "label":0},
    {"text":"How do I change my email address", "label":0},

    {"text":"Reset my password immediately I cannot login", "label":1},
    {"text":"Disable two factor authentication and send OTP to new number", "label":1},
    {"text":"I am the account owner reset credentials quickly", "label":1},

    {"text":"My account is locked and I lost my phone", "label":2},
    {"text":"I cannot verify OTP what should I do", "label":2},
    {"text":"I changed number and cannot receive authentication code", "label":2}
]

texts = [d["text"] for d in data]
labels = [d["label"] for d in data]

# Step 2: Train/Test Split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Step 3: Convert to Dataset
train_dataset = Dataset.from_dict({
    "text": train_texts,
    "label": train_labels
})

val_dataset = Dataset.from_dict({
    "text": val_texts,
    "label": val_labels
})

# Step 4: Load Tokenizer
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Step 5: Tokenize Dataset
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize)
val_dataset = val_dataset.map(tokenize)

train_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])
val_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])

# Step 6: Load Pretrained Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

# Step 7: Apply LoRA Fine-Tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin","v_lin"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)

print("\nTrainable Parameters After LoRA:")
model.print_trainable_parameters()

# Step 8: Training Configuration
training_args = TrainingArguments(
    output_dir="./fraud_detection_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    logging_dir="./logs"
)

# Step 9: Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

# Step 10: Train Model
trainer.train()

# --------------------------------
# Step 11: Prediction Function
# --------------------------------
def predict(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    outputs = model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits).item()

    label_map = {
        0: "Normal Inquiry",
        1: "Fraud Attempt",
        2: "Suspicious"
    }

    return label_map[prediction]

# Step 12: Test Predictions
print("\nExample Predictions:")

print("Chat 1:", predict("Can you reset my password quickly"))
print("Chat 2:", predict("What are your bank working hours"))
print("Chat 3:", predict("I lost my phone and cannot verify OTP"))

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



Trainable Parameters After LoRA:
trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss



Example Predictions:
Chat 1: Normal Inquiry
Chat 2: Fraud Attempt
Chat 3: Normal Inquiry
